# MIN Signal Operator — 13C Environment to Kernel to Temporal Representation

This notebook consolidates 13C-1, 13C-2 and 13C-3. The experiment scripts remain authoritative; the notebook loads their CSV artifacts, produces research plots, and can regenerate the CSVs.

Scientific chain: environment → covariance → positive SOE kernel → GFE/GGFE geometry → temporal basis geometry → MIN state geometry.


In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
RESULTS = ROOT / 'experiments' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
RUN_EXPERIMENTS = False  # True regenerates all three experiment CSV sets.
scripts = [ROOT/'experiments/13C-1_environment_to_kernel_geometry.py', ROOT/'experiments/13C-2_finite_observation_kernel_uncertainty.py', ROOT/'experiments/13C-3_environment_kernel_model_order_mismatch.py']
if RUN_EXPERIMENTS:
    for script in scripts:
        subprocess.run([sys.executable, str(script)], cwd=ROOT, check=True)
    print('CSV artifacts regenerated.')
else:
    print('Using existing CSV artifacts. Set RUN_EXPERIMENTS=True to regenerate.')


In [ ]:
df1 = pd.read_csv(RESULTS/'13C-1_environment_to_kernel_geometry_results.csv')
df2 = pd.read_csv(RESULTS/'13C-2_finite_observation_kernel_uncertainty_results.csv')
df3 = pd.read_csv(RESULTS/'13C-3_environment_kernel_model_order_mismatch_results.csv')
print(df1.shape, df2.shape, df3.shape)


## 13C-1 — Environment to kernel geometry
The first two plots show the upstream mapping and the resulting environment-dependent weighted representation.

In [ ]:
s1=df1.groupby('environment',as_index=False).agg(env_integral=('environment_correlation_integral','mean'),m_scale=('gfe_m_scale_decades','mean'),basis=('weighted_basis_participation_dimension','mean'),state=('weighted_state_participation_dimension','mean'))
fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(s1.env_integral,s1.m_scale,s=70)
for _,r in s1.iterrows(): ax.annotate(r.environment,(r.env_integral,r.m_scale),xytext=(5,5),textcoords='offset points')
ax.set(xlabel='Environment correlation integral',ylabel='Fitted M_scale (decades)',title='13C-1: environment scale to fitted kernel scale'); ax.grid(alpha=.25); plt.show()


In [ ]:
x=np.arange(len(s1)); fig,ax=plt.subplots(figsize=(9,5)); w=.36
ax.bar(x-w/2,s1.basis,w,label='Weighted basis participation'); ax.bar(x+w/2,s1.state,w,label='Weighted MIN state participation')
ax.set_xticks(x,s1.environment,rotation=25,ha='right'); ax.set_ylabel('Participation dimension'); ax.set_title('13C-1: environment-dependent temporal geometry'); ax.legend(); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.show()


## 13C-2 — Finite observation uncertainty
These plots keep kernel identification error separate from representation error.

In [ ]:
s2=df2.groupby(['observation_length','snr_db'],as_index=False).agg(kernel=('kernel_relative_l2_error','mean'),state=('weighted_state_participation_relative_error','mean'))
fig,ax=plt.subplots(figsize=(9,5))
for snr,g in s2.groupby('snr_db'): ax.plot(g.observation_length,g.state,marker='o',label=f'{snr:g} dB')
ax.set_xscale('log',base=2); ax.set(xlabel='Environment observation length (samples)',ylabel='Weighted-state participation relative error',title='13C-2: representation uncertainty'); ax.legend(title='SNR'); ax.grid(alpha=.25); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
for snr,g in s2.groupby('snr_db'): ax.plot(g.observation_length,g.kernel,marker='o',label=f'{snr:g} dB')
ax.set_xscale('log',base=2); ax.set(xlabel='Environment observation length (samples)',ylabel='Kernel relative error',title='13C-2: kernel identification uncertainty'); ax.legend(title='SNR'); ax.grid(alpha=.25); plt.show()


## 13C-3 — Model-order and rate-support mismatch
The important comparison is dictionary/rate support against realized state geometry, not mode count alone.

In [ ]:
s3=df3.groupby('dictionary',as_index=False).agg(kernel=('kernel_fit_relative_l2','mean'),state=('weighted_state_relative_error','mean'),modes=('dictionary_mode_count','first'))
fig,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(s3)); ax.bar(x,s3.state); ax.set_xticks(x,s3.dictionary,rotation=20,ha='right'); ax.set_ylabel('Weighted-state participation relative error'); ax.set_title('13C-3: dictionary mismatch to temporal-state geometry'); ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(9,5)); ax.scatter(s3.kernel,s3.state,s=80)
for _,r in s3.iterrows(): ax.annotate(r.dictionary,(r.kernel,r.state),xytext=(5,5),textcoords='offset points')
ax.set(xlabel='Kernel fit relative L2 error',ylabel='Weighted-state participation relative error',title='13C-3: kernel fidelity versus representation fidelity'); ax.grid(alpha=.25); plt.show()


## Dimension hierarchy

L is nominal SOE mode count.

D_eff = L exp(H_mem) is the GGFE entropy-effective count.

D_basis is realized finite-horizon SOE basis geometry.

D_state is realized MIN temporal-state geometry.

D_task is reserved for a later task-relevance experiment. These quantities are not interchangeable.


In [ ]:
export_rows=[]
for _,r in s1.iterrows(): export_rows.append({'experiment':'13C-1','environment':r.environment,'env_integral':r.env_integral,'m_scale':r.m_scale,'basis_dim':r.basis,'state_dim':r.state})
for _,r in s2.iterrows(): export_rows.append({'experiment':'13C-2','observation_length':r.observation_length,'snr_db':r.snr_db,'kernel_error':r.kernel,'state_error':r.state})
for _,r in s3.iterrows(): export_rows.append({'experiment':'13C-3','dictionary':r.dictionary,'modes':r.modes,'kernel_fit_error':r.kernel,'state_error':r.state})
out=RESULTS/'13C_combined_notebook_summary.csv'; pd.DataFrame(export_rows).to_csv(out,index=False); print('Wrote',out)


## Reproducibility boundary

The notebook is a synthesis/presentation layer. The three experiment scripts remain the numerical source of truth. Setting RUN_EXPERIMENTS to True regenerates their CSV outputs; the notebook then reads those files and produces a compact combined CSV summary.
